# Conversion to ONNX

The image size with dependencies, the model weights file, and the runtime memory combined makes this load too slow for Sagemaker Serverless. The deployment health check is timing out. To resolve this issue, this converts the model file to ONNX for a smaller deployment size and faster startup.

## Prerequisites

A model file downloaded from [small-animal-classifier](https://github.com/agentmorris/small-animal-classifier/releases/tag/v1.0).

In [2]:
import torch
import timm
import time

## Load the checkpoint

In [6]:
# This is the path to the downloaded model artifact
checkpoint_path = "./model-weights/eva02-20260630-llrd.best.e02-s053514.stripped.pt"

ck = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

## Rebuild the Model

In [9]:
model = timm.create_model(
    ck["model_name"],
    pretrained=False,
    num_classes=ck["num_classes"]
)
model.load_state_dict(ck["state_dict"])
model.eval()

print(f"Model: {ck["model_name"]}, classes: {ck["num_classes"]}, input: {ck["img_size"]}x{ck["img_size"]}")

Model: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k, classes: 29, input: 448x448


## Export to ONNX

In [13]:
img_size = ck["img_size"]
dummy_input = torch.randn(1, 3, img_size, img_size)

torch.onnx.export(
    model,
    dummy_input,
    "model-weights/small-animal-classifier.onnx",
    input_names=["input"],
    output_names=["logit"],
    opset_version=17,
    dynamo=True
)

print("Export complete")

W0817 09:41:57.518000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0817 09:41:57.519000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0817 09:41:57.520000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0817 09:41:57.521000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Eva([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Eva([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Conversion to opset < 18 is not supported.


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Export complete


## Quick Sanity Check of ONNX Model

In [14]:
import onnxruntime as ort
import numpy as np

In [15]:
session = ort.InferenceSession("model-weights/small-animal-classifier.onnx")

2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.795197 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6914'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796434 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6899'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796449 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6877'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796458 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6843'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:2813983

In [16]:
test_input = torch.randn(1, 3, img_size, img_size)

with torch.no_grad():
    pt_logits = model(test_input).numpy()

ort_logits = session.run(None, {"input": test_input.numpy()})[0]

max_diff = np.abs(pt_logits - ort_logits).max()
print(f"Max absolute difference: {max_diff:.8f}")
print(f"Match {"yes" if max_diff < 1e-4 else "no"}")

Max absolute difference: 0.00000536
Match yes
